## MLR F-test and R-squared

In this notebook we take a look at the different hypothesis tests that we can make about the model coefficients, including the different types of partial F-tests and t-tests. We also take a look at the adjusted $R^2$.

**What this notebook covers**

1. **Preparing the data.** The automobile file writes missing values as `?` and
   stores some numbers as text, so there is cleaning to do before anything will fit.
2. **Fitting an MLR model** and reading the parts of `.summary()`.
3. **Three different F-tests**, and why they disagree:
   - the **global** F-test -- is the model better than no model at all?
   - **sequential** partial F-tests (`typ=1`) -- depend on the order you list predictors;
   - **marginal** partial F-tests (`typ=2`) -- each predictor given *all* the others.
4. Checking by hand that every ANOVA `sum_sq` really is the difference between two models' SSE.
5. Using **adjusted $R^2$** to compare candidate models, on the Credit data.

The running theme: these tests answer *different questions*, so they can point at
different models. None of them is "the" right answer on its own.


In [1]:
import matplotlib.pyplot as plt      # plotting
import pandas as pd                  # data frames
import numpy as np                   # np.nan for missing values
import statsmodels.api as sm         # sm.stats.anova_lm -> the ANOVA tables
import statsmodels.formula.api as smf  # smf.ols -> fit a model from an R-style formula


<b>Example: automobile data</b>

https://www.kaggle.com/toramky/automobile-dataset

### Load the data

`.sample(5)` shows five *random* rows rather than the first five. That is a better
first look: problems in a file often start partway down, where `.head()` would not
see them.


In [2]:
carsdata=pd.read_csv('../data/Automobile_data.csv')
carsdata.sample(5)   # five random rows -- re-run to draw a different five

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
31,2,137,honda,gas,std,two,hatchback,fwd,front,86.6,...,92,1bbl,2.91,3.41,9.2,76,6000,31,38,6855
157,0,91,toyota,gas,std,four,hatchback,fwd,front,95.7,...,98,2bbl,3.19,3.03,9.0,70,4800,30,37,7198
72,3,142,mercedes-benz,gas,std,two,convertible,rwd,front,96.6,...,234,mpfi,3.46,3.1,8.3,155,4750,16,18,35056
133,2,104,saab,gas,std,four,sedan,fwd,front,99.1,...,121,mpfi,3.54,3.07,9.3,110,5250,21,28,12170
152,1,74,toyota,gas,std,four,hatchback,fwd,front,95.7,...,92,2bbl,3.05,3.03,9.0,62,4800,31,38,6488


To fit a model to find important predictors of `price`, let's start with some predictors to fit an initial model.


### Keep the columns we need, and rename them

Two separate things happen in the next cell.

- We keep only the four columns we plan to model, and `.copy()` so that later edits act on a genuine new frame rather than a view of `carsdata`.
- We **rename the columns to drop the hyphens.** In a formula, `-` means "remove this term", so `'price ~ engine-size'` would be read as *engine minus size* and fail.


In [3]:
cars=carsdata[['engine-size','horsepower','city-mpg','price']].copy()
# renaming columns to remove hyphens and make them easier to work with
cars.columns = ['enginesize', 'horsepower', 'citympg', 'price']


### The missing values are written as `?`

pandas has no way to know that `?` means "missing", so it reads any column
containing one as text (`object`) and `.isnull()` reports nothing wrong. We
convert `?` to `np.nan` first, and only then count.


In [4]:
#deal with missing values: the missing values here are "?"
cars['enginesize'] = cars['enginesize'].replace('?', np.nan)
cars['horsepower'] = cars['horsepower'].replace('?', np.nan)
cars['citympg']   = cars['citympg'].replace('?', np.nan)
cars['price']     = cars['price'].replace('?', np.nan)
cars.isnull().sum()   # now the NaNs are visible and countable


enginesize    0
horsepower    2
citympg       0
price         4
dtype: int64

### Drop the incomplete rows

This is *listwise deletion*: any row missing any one of the four variables is
discarded whole. That is defensible here because only a few rows are affected,
but note two things -- it throws away the rest of the information in those rows,
and it is only safe if the missingness is unrelated to `price`. Imputation (from
the EDA/Comms course assuming it was covered) is the alternative.


In [5]:
#Not too many missing values, for now let's drop them
#Recall how we could impute missing values from EDA
carsnew=cars.dropna().copy()
carsnew.info()          # note the dtypes -- two are still 'object'
carsnew.isnull().sum()


<class 'pandas.DataFrame'>
Index: 199 entries, 0 to 204
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   enginesize  199 non-null    int64
 1   horsepower  199 non-null    str  
 2   citympg     199 non-null    int64
 3   price       199 non-null    str  
dtypes: int64(2), str(2)
memory usage: 7.8 KB


enginesize    0
horsepower    0
citympg       0
price         0
dtype: int64

### Fix the data types

`price` and `horsepower` are still `object`, because the `?` values forced pandas
to read those columns as text. This has to be fixed before fitting, and the reason
is worth knowing: given an `object` column, a formula treats it as
**categorical** and silently fits one dummy variable (which we will learn about later) per distinct value. You would
get a hundred-odd dummies instead of a single slope, and the ANOVA table would be
meaningless.


In [6]:
#one more problem: price is a object, not int64(numeric), so is horsepower!
carsnew['horsepower'] = carsnew['horsepower'].astype('int64')
carsnew['price'] = carsnew['price'].astype('int64')
carsnew.info()          # all four numeric now -- safe to fit


<class 'pandas.DataFrame'>
Index: 199 entries, 0 to 204
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   enginesize  199 non-null    int64
 1   horsepower  199 non-null    int64
 2   citympg     199 non-null    int64
 3   price       199 non-null    int64
dtypes: int64(4)
memory usage: 7.8 KB


### Fit a model and examine the results

Now we can fit mlr: price~enginesize + citympg+ horsepower

`smf.ols` takes an R-style formula: response on the left of `~`, predictors on the
right joined by `+`. The intercept is included automatically. `.fit()` runs the
least squares computation and hands back a *results* object -- that object is what
carries `.summary()`, `.ssr` (the SSE), `.rsquared`, `.params`, and the rest.


In [7]:
# Fit the model
# price is the response; the three predictors are added with '+'
reg = smf.ols('price~enginesize+citympg+horsepower',data=carsnew).fit()


### Reading the summary

Three blocks are worth your attention.

- **Top right** -- `R-squared` and `Adj. R-squared`, then `F-statistic` with
  `Prob (F-statistic)`. That F is the **global** test, $H_0$: every slope is zero,
  i.e. the model does no better than predicting with $\bar{y}$.
- **Middle table** -- one row per coefficient: the estimate, its standard error,
  and the **t-test** of $H_0: \beta_j = 0$ *given that every other predictor is
  already in the model*. This is a marginal test, not a test of the predictor on
  its own.
- **Bottom** -- diagnostics (skew, kurtosis, Durbin--Watson, condition number),
  which we return to when we cover model diagnostics.


In [8]:
#Model summary
reg.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.794
Model:                            OLS   Adj. R-squared:                  0.791
Method:                 Least Squares   F-statistic:                     251.2
Date:                Mon, 21 Sep 2026   Prob (F-statistic):           1.03e-66
Time:                        14:00:51   Log-Likelihood:                -1912.4
No. Observations:                 199   AIC:                             3833.
Df Residuals:                     195   BIC:                             3846.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -2547.0951   2914.668     -0.874      0.383   -8295.415    3201.225
enginesize   124.3388     10.951     11.355      0.000     102.742     145.935
citympg     -151.3184     70.849     -2.136      0.034    -291.047     -11.590
horsepower    37.0876     16.262      2.281      0.024       5.016      69.159
==============================================================================
Omnibus:                       10.990   Durbin-Watson:                   0.771
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               17.130
Skew:                           0.312   Prob(JB):                     0.000191
Kurtosis:                       4.295   Cond. No.                     1.96e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.96e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

#### Anova F-test - WILL BE ON FINAL

- from global anova: the model is significant compared to the null model. This is not a very useful result.

- from t-test: enginesize, citympg, horsepower are all significant predictors of price, given the other predictors are in the model. 

When we extract `anova_lm` separately, we can use different arguments for `typ` to perform different types of partial F-tests. We can use `typ=1` to perform something called a *sequential F-test*.

#### Sequential 

A sequential model is one where we add predictors one at a time and see how the model improves. This is useful for understanding the contribution of each predictor to the model, but unfortunately, the order matters. So, in the table below, the `sum_sq` column is showing us the reduction in the sum of squares as you add each predictor in sequence. The `Residual` row is the SSE for the full model with each of the predictors in it. The p-value tells us if adding the predictor to a model that already contains the previous predictors (the ones higher in the table) significantly reduces the sum of the squared residuals.

How to read the columns below: `df` is 1 for each predictor (one slope added),
`sum_sq` is the drop in SSE from adding that predictor **to the ones listed above
it**, `mean_sq` is `sum_sq / df`, and `F` is that `mean_sq` divided by the
`Residual` `mean_sq` (which is the full model's MSE). The `Residual` row holds the
full model's SSE on $n - p$ degrees of freedom.


In [9]:
# typ=1 and typ=2 in anova_lm gives different values in SS and F test
sm.stats.anova_lm(reg, typ=1) #sequential


,df,sum_sq,mean_sq,F,PR(>F)
enginesize,1.0,9.625888e+09,9.625888e+09,724.441229,1.387785e-67
citympg,1.0,3.186064e+08,3.186064e+08,23.978216,2.039654e-06
horsepower,1.0,6.911228e+07,6.911228e+07,5.201368,2.365037e-02
Residual,195.0,2.591029e+09,1.328733e+07,NaN,NaN


#### Partial

If we choose `typ=2` we perform the partial F-test where the full model is compared to a model with one predictor dropped. For example, in the first row of the table below we see the result of comparing the reduced model `price ~ citympg + horsepower` -- `enginesize` dropped -- against the full `price ~ enginesize + citympg + horsepower`. Each row does the same thing for a different predictor, and crucially every row is compared against the *same* full model. That is why these rows do not depend on the order the predictors are listed in.


In [ ]:
sm.stats.anova_lm(reg, typ=2) #partial

#h0: reduced model with the specific predictor dropped
#h1: always the full model 

# each row is independent, they don't build off of each other


,sum_sq,df,F,PR(>F)
enginesize,1.713088e+09,1.0,128.926426,2.904006e-23
citympg,6.061127e+07,1.0,4.561585,3.394397e-02
horsepower,6.911228e+07,1.0,5.201368,2.365037e-02
Residual,2.591029e+09,195.0,NaN,NaN


`typ=3` returns the same sums of squares as `typ=2` here, plus an extra row testing the intercept. For a model like this one, with no categorical predictors and no interactions, types 2 and 3 agree. They can differ once you have unbalanced factors.


In [11]:
# typ-3 is the same as typ=2, just added a test on intercept
sm.stats.anova_lm(reg, typ=3)

,sum_sq,df,F,PR(>F)
Intercept,1.014728e+07,1.0,0.763681,3.832538e-01
enginesize,1.713088e+09,1.0,128.926426,2.904006e-23
citympg,6.061127e+07,1.0,4.561585,3.394397e-02
horsepower,6.911228e+07,1.0,5.201368,2.365037e-02
Residual,2.591029e+09,195.0,NaN,NaN


## Typ=1, checked by hand

The claim behind the sequential table is that each `sum_sq` is literally the
difference between two models' SSE. Let's check it for `citympg`, which sits
*second* in the order `enginesize + citympg + horsepower`, so the two models being
compared are

- reduced: `price ~ enginesize`
- full: `price ~ enginesize + citympg`

and the sequential SS for `citympg` should come out as SSE(reduced) $-$ SSE(full).


In [12]:
#Take a closer look at typ=1: reduced model without citympg
reg_reduced_1 = smf.ols('price~enginesize',data=carsnew).fit()
sm.stats.anova_lm(reg_reduced_1, typ=1)   # read the Residual row: SSE of the reduced model


,df,sum_sq,mean_sq,F,PR(>F)
enginesize,1.0,9.625888e+09,9.625888e+09,636.609809,1.265067e-63
Residual,197.0,2.978748e+09,1.512055e+07,NaN,NaN


In [13]:
#Take a closer look at typ=1: full model with citympg
reg_full_1 = smf.ols('price~enginesize+citympg',data=carsnew).fit()
sm.stats.anova_lm(reg_full_1, typ=1)      # Residual row again: SSE once citympg is added


,df,sum_sq,mean_sq,F,PR(>F)
enginesize,1.0,9.625888e+09,9.625888e+09,709.238315,4.852522e-67
citympg,1.0,3.186064e+08,3.186064e+08,23.475016,2.566218e-06
Residual,196.0,2.660141e+09,1.357215e+07,NaN,NaN


In [14]:
## SS(Resid; enginesize only) - SS(Resid; enginesize+citympg) =
2.978748e+09 - 2.660141e+09
# the two numbers are pasted from the Residual rows above.
# Equivalent, and won't go stale if the data ever changes:
#     reg_reduced_1.ssr - reg_full_1.ssr


318607000.0

The difference in SSE is the same as the SS for citympg in y~enginesize+citympg+horsepower

In [ ]:
sm.stats.anova_lm(reg, typ=1)   # compare the citympg row with the difference computed above


But, changing the predictor order in typ=1 alters the result because you are comparing different reduced and full models now

In [ ]:
reg_order= smf.ols('price~citympg+horsepower+enginesize',data=carsnew).fit()
sm.stats.anova_lm(reg_order, typ=1)   # same three predictors, different order


Three things to notice between the two orderings:

- **`horsepower`'s row changed completely.** In the first table it was added last; here it is added second, so it is being credited with variation that
  `enginesize` had already claimed before.
- **The `Residual` row is identical**, and the predictor sums of squares still add up to the same SSR. 
- **The last predictor listed always matches its `typ=2` value.** A predictor that is added to the model last, or compared against all of the other predictors, describe the same comparison. Check it: `enginesize` is last here, and its `sum_sq` equals `enginesize` in the `typ=2` table above.


<b> From sequential F tests </b>

- We can kind of "rank" the importance of the predictors. 
- Since in the above example all predictors are significant, we will leave this for the credit data example. 

## Typ=2, checked by hand

Same exercise for the partial table. Here every row drops one predictor from the
*same* full model, so for `citympg` the comparison is

- reduced: `price ~ enginesize + horsepower`
- full: `price ~ enginesize + citympg + horsepower`


In [ ]:
#Take a closer look at typ=2: reduced model without citympg
reg_reduced_2 = smf.ols('price~enginesize+horsepower',data=carsnew).fit()
sm.stats.anova_lm(reg_reduced_2, typ=2)   # Residual row = SSE with citympg removed


In [ ]:
#full
sm.stats.anova_lm(reg, typ=2)   # Residual row = SSE of the full model


In [ ]:
#citympg	6.061127e+07	 = Residual(reduced) 2.651640e+09 - Residual(full)	2.591029e+09	
2.651640e+09 - 2.591029e+09 #approx bc of sigfig issue

The difference in SSE (SS Residual) between reduced and full models is the SS of citympg in the full model!!

Similar story if we check other reduced models with dropped engine or horsepower:

In [ ]:
reg_reduced_noengine = smf.ols('price~citympg+horsepower',data=carsnew).fit()
sm.stats.anova_lm(reg_reduced_noengine, typ=2)   # drop enginesize instead


In [ ]:
4.304117e+09 - 2.591029e+09 #enginesize	1.713088e+09	
# = SS(engine) in full model

In [ ]:
reg_reduced_nohorse = smf.ols('price~citympg+enginesize',data=carsnew).fit()
sm.stats.anova_lm(reg_reduced_nohorse, typ=2)    # drop horsepower instead


In [ ]:
2.660141e+09 - 2.591029e+09  #horsepower	~6.911228e+07
# = SS(horsepower) in full model

<b> From partial F tests </b>

- Every reduced model is compared to the same full model:

For example, the F test for `enginesize` is comparing 

- Reduced model: price ~ citympg+ horsepower
- Full model: price ~ enginesize+ citympg+ horsepower

So it will suggest the significance of one predictor conditional on the fact ALL OTHER predictors are already in the model. We can use it to 

- judge the "conditional" importance of each predictor
- pick a reduced model with one less predictor that's better than the full model. 

<b>Example: Credit data</b>

### A second example, where the tests disagree

On the automobile data all three predictors were significant however you asked,
so the different tests never actually conflicted. The Credit data is more
interesting: the t-tests, the sequential F-tests and the partial F-tests point at
*different* models, which is the whole reason for knowing the difference between
them.


In [15]:
# example: Credit data
credit = pd.read_csv("../data/Credit.csv")
credit.info()   # check the dtypes before fitting, same as before


<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  400 non-null    int64  
 1   Income      400 non-null    float64
 2   Limit       400 non-null    int64  
 3   Rating      400 non-null    int64  
 4   Cards       400 non-null    int64  
 5   Age         400 non-null    int64  
 6   Education   400 non-null    int64  
 7   Gender      400 non-null    str    
 8   Student     400 non-null    str    
 9   Married     400 non-null    str    
 10  Ethnicity   400 non-null    str    
 11  Balance     400 non-null    int64  
dtypes: float64(1), int64(7), str(4)
memory usage: 37.6 KB


`Income` arrives as text, so it needs converting before it can be a numeric
predictor -- the same trap as `price` and `horsepower` earlier.


In [16]:
credit['Income'] = pd.to_numeric(credit['Income'])   # text -> numeric, or it would be treated as categorical


In [ ]:
model =smf.ols('Balance~Income + Limit + Rating + Age',data=credit).fit()
model.summary()   # look at the individual t-tests (P>|t|) in the coefficient table


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                Balance   R-squared:                       0.877
Model:                            OLS   Adj. R-squared:                  0.876
Method:                 Least Squares   F-statistic:                     705.6
Date:                Mon, 21 Sep 2026   Prob (F-statistic):          2.16e-178
Time:                        14:22:09   Log-Likelihood:                -2599.9
No. Observations:                 400   AIC:                             5210.
Df Residuals:                     395   BIC:                             5230.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -445.1048     40.576    -10.970      0.000    -524.877    -365.332
Income        -7.6127      0.382    -19.945      0.000      -8.363      -6.862
Limit          0.0818      0.045      1.834      0.067      -0.006       0.170
Rating         2.7314      0.664      4.111      0.000       1.425       4.038
Age           -0.8561      0.478     -1.789      0.074      -1.797       0.084
==============================================================================
Omnibus:                       94.733   Durbin-Watson:                   1.906
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              165.919
Skew:                           1.374   Prob(JB):                     9.36e-37
Kurtosis:                       4.550   Cond. No.                     2.65e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.65e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

From individual t-testing, Income and Rating are significant, while Limit and Age are on the boundary. So t-tests suggests the model: y~Income+Rating

Now the sequential table, **using the same order the formula was written in**:
`Income`, then `Limit`, then `Rating`, then `Age`. Each row asks whether that
predictor earns its place given only the ones above it.


In [18]:
# try sequential anova with the same order

sm.stats.anova_lm(model, typ=1)   # order here is Income, Limit, Rating, Age



,df,sum_sq,mean_sq,F,PR(>F)
Income,1.0,1.813117e+07,1.813117e+07,691.691426,7.902872e-89
Limit,1.0,5.533791e+07,5.533791e+07,2111.102870,1.465723e-160
Rating,1.0,4.328357e+05,4.328357e+05,16.512381,5.832806e-05
Age,1.0,8.394134e+04,8.394134e+04,3.202304,7.430058e-02
Residual,395.0,1.035406e+07,2.621280e+04,NaN,NaN


Here:
- p-values disagree with t tests
- Suggests y\~Income+Limit is sig. better than just y\~Income, and  y\~Income+Limit+Rating is sig. better than y\~Income+Limit
- y~ Income+Limit+ Rating is the best model according to seq. F-test (since adding age is not significant; 7.430058e-02>0.05)

The order is doing a lot of work above, so the natural check is to move the
doubtful predictor. `Limit` goes last, which -- by the point made earlier -- turns
its sequential row into the same comparison as its partial (`typ=2`) row.


In [ ]:
#Now consider putting the questionable one (Limit) at the end
model_2 =smf.ols('Balance~Age + Income + Rating + Limit',data=credit).fit()
sm.stats.anova_lm(model_2, typ=1)   # Limit is last now, so this row = its typ=2 row


Now the sequential anova suggests y~Income+Rating might be enough, adding Limit doesn't significantly improve it anymore.

- limit might be on the boundary

<b>Can you try typ=2?</b> What does that tell us?

For the candidates, let's look at R^2 and adj-R^2

### Adjusted $R^2$

$R^2$ cannot fall when you add a predictor, so it is useless for choosing between
nested models -- it always prefers the larger one. Adjusted $R^2$ divides each sum
of squares by its degrees of freedom,

$$R^2_a = 1 - \frac{SSE/(n-p)}{SST/(n-1)},$$

so a predictor that does not reduce SSE enough to pay for the degree of freedom it
costs will *lower* it. That is what makes it comparable across models of different
size.

The four candidates below come from the three different analyses above, which is
the point of collecting them here.


In [ ]:
model_c1 = smf.ols('Balance~Income + Rating ',data=credit).fit() #from t test and anova(typ=2)
model_c2 = smf.ols('Balance~Income + Rating + Limit',data=credit).fit()#from anova(typ=1)
model_c3 = smf.ols('Balance~Income + Rating + Age',data=credit).fit() #not that strong
model_c4 = smf.ols('Balance~Income + Rating + Age + Limit',data=credit).fit() #not that strong

In [ ]:
print(model_c1.rsquared, model_c1.rsquared_adj)
print(model_c2.rsquared, model_c2.rsquared_adj)
print(model_c3.rsquared, model_c3.rsquared_adj)
print(model_c4.rsquared, model_c4.rsquared_adj)

# left column: R^2, which rises with every predictor added.
# right column: adjusted R^2, which is the one to compare across models.


Notice the left column: $R^2$ climbs every time a predictor is added, exactly as it
must. That is why it cannot be used to choose between nested models -- the largest
model always wins. The right column is the one to compare across models.


Out of the four candidates, **`model_c4` has the highest adjusted $R^2$** ($0.8760$,
against $0.8753$ for `model_c2`) -- so on this criterion alone the full
four-predictor model wins.

But look at the spread: all four sit within about $0.0015$ of each other. That is not
a real separation. Adjusted $R^2$ will happily keep a predictor that pays for its
degree of freedom by the narrowest margin, so "highest adjusted $R^2$" is a weak
reason to prefer one of these models over another.

Reading the t-tests and the two ANOVA tables together, the evidence ranks the
predictors roughly

$$\text{Income} \;>\; \text{Rating} \;>\; \text{Limit} \;\approx\; \text{Age}$$

(partial $F$ of $397.8$, $16.9$, $3.4$, $3.2$ respectively), with only the first two
clearly significant.


### Remark 

You can already see that different methods lead to different conclusions. After we learn modeling diagnostics and more selection methods, we will consider many model selection criteria, and argue to make a decision based on a specific one which makes sense in terms of interpretation and prediction for the problem at hand; if still hard to decide, you can try the prediction on a test set to go with the one with better prediction performance. 

